# 🔌 Using Commercial LLMs with LangChain

## Learning Objectives
In this notebook, you will learn:
1. **OpenAI through LangChain** - build a `prompt | model` chain with `ChatOpenAI`
2. **The `AIMessage` response** - where the text, token usage and model metadata live
3. **Two everyday shortcuts** - `StrOutputParser` for plain strings, and invoking each step by hand
4. **Swapping providers** - run the identical chain on Google Gemini with `ChatGoogleGenerativeAI`

## Where this fits
`02_Commercial_LLMs_Natively.ipynb` called OpenAI and Gemini through their own SDKs. Here the
same providers sit behind LangChain's common chat-model interface. Prompt templates are covered
in depth in `modules/02_inputs_outputs_prompts/notebooks/03_Prompt_Templates.ipynb`,
and chain composition in `modules/03_lcel_and_runnables/`.

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- `GOOGLE_API_KEY` in the project-root `.env` (Part 4, optional)

---
## 📦 Step 1: Environment Setup

LangChain's provider classes read `OPENAI_API_KEY` and `GOOGLE_API_KEY` from the environment, so loading `.env` is all the setup needed.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and GOOGLE_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

---
## 🤖 Part 1: OpenAI ChatGPT with LangChain

The fundamental LangChain pattern: a prompt template with a placeholder, a chat model, and the
`|` operator joining them into a chain you `.invoke()` with a dict of variables.

In [ ]:
# ============================================================================
# BASIC LANGCHAIN CHAIN WITH OPENAI GPT
# ============================================================================
# This demonstrates the fundamental pattern of LangChain:
# 1. Create a prompt template with placeholders (e.g., {topic})
# 2. Initialize an LLM (ChatOpenAI in this case)
# 3. Chain them together using the pipe (|) operator (LCEL syntax)
# 4. Invoke the chain with input variables
# ============================================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Initialize the ChatOpenAI model
# - model: Specifies which GPT model to use (gpt-4o-mini is cost-effective)
# - temperature: Controls randomness (0 = deterministic, 1 = creative)
# The provider class is used directly here on purpose: this notebook is about LangChain's
# provider integrations. (The rest of the course uses aied_helpers.get_openai_llm,
# which returns this same ChatOpenAI class.)
chatgpt = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0  # Use 0 for consistent, reproducible outputs
)

# Create a prompt template with a placeholder variable {topic}
# ChatPromptTemplate is designed for chat models (message-based)
prompt = "Explain {topic} in 2 bullet points"
prompt_template = ChatPromptTemplate.from_template(prompt)

# Create a chain using LCEL (LangChain Expression Language)
# The | operator pipes the output of one component to the next:
# prompt_template -> formats the prompt -> chatgpt -> generates response
chain = (
    prompt_template 
    | 
    chatgpt
)

# Invoke the chain with actual values for the placeholders
# Returns an AIMessage object containing the response
response = chain.invoke({"topic": "AI"})

# Access the text content from the AIMessage object
print(response.content)

---
## 🔍 Part 2: Read the `AIMessage`

The chain returned an `AIMessage`, not a string. Besides `.content` it carries the provider's
metadata — token counts and the model that answered — in a provider-neutral shape.

In [ ]:
# ============================================================================
# EXAMINING THE RESPONSE OBJECT
# ============================================================================
# - content: the text reply
# - response_metadata: provider details (model name, finish reason, ...)
# - usage_metadata: token counts in a provider-neutral format
# ============================================================================
print(f"📋 Type          : {type(response).__name__}")
print(f"📋 Content       : {response.content[:80]}...")
print(f"📋 Model         : {response.response_metadata.get('model_name')}")
print(f"📋 Finish reason : {response.response_metadata.get('finish_reason')}")
print(f"📊 Token usage   : {response.usage_metadata}")

---
## ✂️ Part 3: Two everyday shortcuts

### 3.1 `StrOutputParser`: get the string directly
Appending a parser turns the `AIMessage` into a plain string, so there is no `.content` to reach for.

In [ ]:
# ============================================================================
# ALTERNATIVE 1: Using StrOutputParser for Direct String Output
# ============================================================================
# StrOutputParser extracts the text content from AIMessage automatically
# This is useful when you only need the text and want cleaner code
# Chain flow: prompt -> LLM -> AIMessage -> StrOutputParser -> string
# ============================================================================

from langchain_core.output_parsers import StrOutputParser


# Adding StrOutputParser() to the chain extracts just the string content
# No need to access .content on the response - it's already a string!
chain = prompt_template | chatgpt | StrOutputParser()
response = chain.invoke({"topic": "AI"})
print(response)  # Direct string output - no .content needed!

### 3.2 Invoke each step by hand
A chain is only a sequence of steps. Calling them one by one gives you the intermediate
values — useful for debugging a prompt.

In [ ]:
# ============================================================================
# ALTERNATIVE 5: Direct Invocation Without Chain
# ============================================================================
# Instead of creating a chain, you can invoke each component separately
# This gives you more control and visibility into intermediate results
# Useful for debugging or when you need to inspect/modify data between steps
# ============================================================================

# Step 1: Format the prompt with variables -> produces ChatPromptValue (messages)
messages = prompt_template.invoke({"topic": "AI"})
print(f"📋 Formatted prompt: {messages.to_messages()}")

# Step 2: Send messages to the LLM -> produces AIMessage
response = chatgpt.invoke(messages)
print(response.content)

### 3.3 What `|` builds
`prompt_template | chatgpt` creates a `RunnableSequence`; `prompt_template.pipe(chatgpt)` and
`RunnableSequence(prompt_template, chatgpt)` build the same thing. The cell below checks that
without calling the model. Module 03 (`modules/03_lcel_and_runnables/`) covers composition in depth.

In [ ]:
# ============================================================================
# THREE SPELLINGS, ONE OBJECT TYPE (no model call)
# ============================================================================
from langchain_core.runnables import RunnableSequence

spellings = {
    "prompt | llm": prompt_template | chatgpt,
    "prompt.pipe(llm)": prompt_template.pipe(chatgpt),
    "RunnableSequence(prompt, llm)": RunnableSequence(prompt_template, chatgpt),
}
for name, chain_obj in spellings.items():
    print(f"📋 {name:<30} -> {type(chain_obj).__name__} with {len(chain_obj.steps)} steps")

---
## 🌐 Part 4: Google Gemini with LangChain

> Optional: runs only when `GOOGLE_API_KEY` is set in `.env`.

Only the model class changes — `ChatGoogleGenerativeAI` from `langchain-google-genai`. The
prompt, the chain and `.invoke()` are identical to Part 1.

In [ ]:
# ============================================================================
# USING GOOGLE GEMINI WITH LANGCHAIN
# ============================================================================
# LangChain provides a unified interface - the code structure is identical
# to OpenAI! Only the model class changes:
# - OpenAI: ChatOpenAI from langchain_openai
# - Gemini: ChatGoogleGenerativeAI from langchain_google_genai
#
# This abstraction is one of LangChain's key benefits - easily swap models!
# ============================================================================

import os

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini model
# Current models: https://ai.google.dev/gemini-api/docs/models
# Flash models are faster and cheaper, Pro models are more capable
# Optional provider: runs only when GOOGLE_API_KEY is set
if os.getenv("GOOGLE_API_KEY"):
    gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

    # Same prompt template syntax works across all LLMs
    PROMPT = "Explain {topic} in 2 bullets"
    prompt = ChatPromptTemplate.from_template(PROMPT)

    # Same chain pattern - this is the power of LangChain's abstraction!
    chain = prompt | gemini

    # Invoke works exactly the same way
    response = chain.invoke({"topic": "AI"})
    print(response.content)
else:
    print("⏭️  GOOGLE_API_KEY not set - Gemini section skipped")

---
## 📝 Summary

In this notebook, we learned:

### 1. OpenAI through LangChain
- `prompt_template | chat_model` builds a chain; `.invoke({"key": "value"})` runs it

### 2. The `AIMessage` response
- The reply is an `AIMessage`: text on `.content`, tokens on `.usage_metadata`, provider details on `.response_metadata`

### 3. Two everyday shortcuts
- `StrOutputParser()` returns a plain string; invoking each step by hand exposes the formatted prompt
- `|`, `.pipe()` and `RunnableSequence(...)` all build the same `RunnableSequence`

### 4. Swapping providers
- Changing provider means changing the model class; the chain code stays the same

### Next Steps
- **`04_Open_Source_LLMs_HF_Transformers.ipynb`**: run an open-source model locally, without any API
- Prompt templates in depth: `modules/02_inputs_outputs_prompts/notebooks/03_Prompt_Templates.ipynb`